In [ ]:
# Test Fine-Tuned Model with Coordinate-Based Format
import sys
import os
import json
import re
import numpy as np
from pathlib import Path

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

print("Imports successful")
print(f"Project root: {project_root}")


In [ ]:
# Load the fine-tuned model
finetuned_model_path = "./final_model"  # Adjust if your model is elsewhere

print("Loading base model...")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-0.6B",
    torch_dtype=torch.float16,
    device_map="auto"
)

print(f"Loading fine-tuned adapter from {finetuned_model_path}...")
ft_model = PeftModel.from_pretrained(base_model, finetuned_model_path)
ft_model.eval()

print("Model loaded successfully")


In [ ]:
# Import helper functions
from mysterydungeonGPT.helpers import coordinates_to_grid, extract_json_from_text

print("Helper functions imported")


In [ ]:
# Create test prompt
test_prompt = "Generate a medium difficulty dungeon with 6 rooms"

print(f"Test prompt: {test_prompt}")
print(f"Prompt tokens: {len(tokenizer.encode(test_prompt, add_special_tokens=False))}")


In [ ]:
# Tokenize prompt with chat template
messages = [
    {"role": "user", "content": test_prompt}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(ft_model.device)

print(f"Input shape: {inputs['input_ids'].shape}")
print(f"Input tokens: {inputs['input_ids'].shape[1]}")


In [ ]:
# Generate map
print("Generating map...")
with torch.no_grad():
    outputs = ft_model.generate(
        **inputs,
        max_new_tokens=2000,  # Reduced for coordinate format (~300-800 tokens expected)
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        repetition_penalty=1.1
    )

# Decode only the generated tokens (exclude input prompt)
input_length = inputs['input_ids'].shape[1]
generated_ids = outputs[0][input_length:]
generated_content = tokenizer.decode(generated_ids, skip_special_tokens=True)

print(f"Generation complete")
print(f"Generated tokens: {len(generated_ids)}")
print(f"Generated length: {len(generated_content)} characters")


In [ ]:
# Extract JSON from generated content
map_json = extract_json_from_text(generated_content)

if map_json:
    print("JSON extracted successfully")
    print(f"Keys: {list(map_json.keys())}")
else:
    print("Failed to extract JSON")


In [ ]:
# Validate and reconstruct map
if map_json:
    print("="*70)
    print("MAP VALIDATION")
    print("="*70)
    
    # Check required fields
    required_fields = ['walkable_tiles', 'player_spawn', 'stairs_spawn', 'width', 'height']
    missing_fields = [f for f in required_fields if f not in map_json]
    
    if missing_fields:
        print(f"WARNING: Missing fields: {missing_fields}")
    else:
        print("All required fields present")
    
    # Get data
    walkable_coords = map_json.get('walkable_tiles', [])
    player_spawn = map_json.get('player_spawn', [0, 0])
    stairs_spawn = map_json.get('stairs_spawn', [0, 0])
    width = map_json.get('width', 32)
    height = map_json.get('height', 32)
    
    print(f"\nWalkable tiles: {len(walkable_coords)} coordinates")
    print(f"Player spawn: {player_spawn}")
    print(f"Stairs spawn: {stairs_spawn}")
    print(f"Map size: {width}x{height}")
    
    # Validate coordinates
    valid_coords = []
    invalid_coords = []
    for coord in walkable_coords:
        if isinstance(coord, list) and len(coord) == 2:
            x, y = coord
            if 0 <= x < width and 0 <= y < height:
                valid_coords.append(coord)
            else:
                invalid_coords.append(coord)
        else:
            invalid_coords.append(coord)
    
    print(f"\nValid coordinates: {len(valid_coords)}")
    if invalid_coords:
        print(f"WARNING: Invalid coordinates: {len(invalid_coords)}")
        print(f"   Examples: {invalid_coords[:5]}")
    
    # Reconstruct grid
    print("\nReconstructing grid from coordinates...")
    full_grid = coordinates_to_grid(valid_coords, width=width, height=height)
    
    # Add spawn points (5x5 areas)
    if player_spawn and len(player_spawn) == 2:
        px, py = player_spawn
        for dy in range(-2, 3):
            for dx in range(-2, 3):
                nx, ny = px + dx, py + dy
                if 0 <= nx < width and 0 <= ny < height:
                    if full_grid[ny, nx] == 1:  # Only on floor tiles
                        full_grid[ny, nx] = 2  # Player spawn
    
    if stairs_spawn and len(stairs_spawn) == 2:
        sx, sy = stairs_spawn
        for dy in range(-2, 3):
            for dx in range(-2, 3):
                nx, ny = sx + dx, sy + dy
                if 0 <= nx < width and 0 <= ny < height:
                    if full_grid[ny, nx] == 1:  # Only on floor tiles
                        full_grid[ny, nx] = 3  # Stairs spawn
    
    print(f"Grid reconstructed: {full_grid.shape}")
    print(f"  Walls (0): {np.sum(full_grid == 0)}")
    print(f"  Floors (1): {np.sum(full_grid == 1)}")
    print(f"  Player spawn (2): {np.sum(full_grid == 2)}")
    print(f"  Stairs spawn (3): {np.sum(full_grid == 3)}")
    
    # Check if spawns are on walkable tiles
    if player_spawn and len(player_spawn) == 2:
        px, py = player_spawn
        if 0 <= px < width and 0 <= py < height:
            if full_grid[py, px] == 0:
                print(f"WARNING: Player spawn at ({px}, {py}) is on a wall!")
    
    if stairs_spawn and len(stairs_spawn) == 2:
        sx, sy = stairs_spawn
        if 0 <= sx < width and 0 <= sy < height:
            if full_grid[sy, sx] == 0:
                print(f"WARNING: Stairs spawn at ({sx}, {sy}) is on a wall!")
    
    print("\nMap reconstruction complete!")
else:
    print("ERROR: Cannot reconstruct map - JSON extraction failed")


In [ ]:
# Visualize the map
if map_json and 'full_grid' in locals():
    import matplotlib.pyplot as plt
    
    # Create color map: 0=black (wall), 1=brown (floor), 2=green (player), 3=red (stairs)
    color_map = np.zeros((height, width, 3), dtype=np.uint8)
    color_map[full_grid == 0] = [0, 0, 0]        # Black walls
    color_map[full_grid == 1] = [139, 69, 19]    # Brown floors
    color_map[full_grid == 2] = [0, 255, 0]      # Green player spawn
    color_map[full_grid == 3] = [255, 0, 0]      # Red stairs spawn
    
    plt.figure(figsize=(10, 10))
    plt.imshow(color_map)
    plt.title(f"Generated Map: {test_prompt}")
    plt.axis('off')
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='black', label='Wall'),
        Patch(facecolor='#8B4513', label='Floor'),
        Patch(facecolor='green', label='Player Spawn'),
        Patch(facecolor='red', label='Stairs Spawn')
    ]
    plt.legend(handles=legend_elements, loc='upper right')
    
    plt.tight_layout()
    plt.show()
    
    print("Map visualization complete")


In [ ]:
# Save map for game use (optional)
if map_json and 'full_grid' in locals():
    # Convert to game format (full grid with tiles array)
    game_map = {
        'tiles': full_grid.tolist(),
        'player_spawn': player_spawn,
        'stairs_spawn': stairs_spawn,
        'width': width,
        'height': height,
        'difficulty': map_json.get('difficulty', 'medium'),
        'enemies': map_json.get('enemies', [])
    }
    
    # Save to file
    maps_dir = Path("../web_game/maps")
    maps_dir.mkdir(parents=True, exist_ok=True)
    
    map_id = "map_generated_test"
    output_path = maps_dir / f"{map_id}.json"
    
    with open(output_path, 'w') as f:
        json.dump(game_map, f, indent=2)
    
    print(f"Map saved to {output_path}")
    print(f"  You can now load this in the web game!")
else:
    print("WARNING: Skipping save - map reconstruction failed")


In [ ]:
# Print summary
print("="*70)
print("TEST SUMMARY")
print("="*70)
print(f"Prompt: {test_prompt}")
print(f"Generated tokens: {len(generated_ids) if 'generated_ids' in locals() else 'N/A'}")
print(f"JSON extracted: {'Yes' if map_json else 'No'}")
print(f"Walkable coordinates: {len(walkable_coords) if map_json else 'N/A'}")
print(f"Grid reconstructed: {'Yes' if 'full_grid' in locals() else 'No'}")
print(f"Map saved: {'Yes' if 'output_path' in locals() else 'No'}")
print("="*70)
